# 🎬 YouTube RAG Chatbot — Q&A Over YouTube Transcripts

**Selected Topic:** Business & Financial Literacy — *How Insurance Companies Work*

| Setting | Value |
|---|---|
| Default Video | [The Infographics Show — Why do insurance companies make money?](https://www.youtube.com/watch?v=qjXgpJpSlCc) |
| Video ID | `qjXgpJpSlCc` |
| LLM Provider | Ollama (`llama3.2`) / Groq fallback (`llama-3.3-70b-versatile`) |
| Embedding Model | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector Store | ChromaDB (cosine similarity, persisted to `chroma_db/`) |

> **How to run:** Execute all cells top to bottom (*Kernel → Restart & Run All*).  
> Set the `GROQ_API_KEY` environment variable if using Groq instead of Ollama.

## Section 0 - Install Dependencies

Installs all third-party libraries needed for the project in one quiet pip command.
Run this cell once; restart the kernel if prompted after installation.

In [1]:
%pip install -q youtube-transcript-api langchain-text-splitters sentence-transformers chromadb ollama groq gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Section 1 - Configuration

Central config cell: every tunable setting for the project lives here.
Edit this cell to change video IDs, model names, chunking strategy, or retrieval thresholds - never hard-code these values elsewhere.

In [2]:
import os
from pathlib import Path

DEFAULT_VIDEO_URL = "qjXgpJpSlCc"                  # the YouTube link used when the user does not paste their own; the video must have captions
TRANSCRIPT_LANGUAGES = ["en", "en-US", "en-GB", "en-CA", "en-AU"]  # caption language; covers major English variants
TRANSCRIPT_DIR = Path("transcripts")               # folder where raw transcripts are saved
CHROMA_DIR = "chroma_db"                           # local directory for the persistent ChromaDB store
COLLECTION_NAME = "yt_transcripts"                 # ChromaDB collection name
CHUNK_SIZE = 800                                   # max characters per text chunk
CHUNK_OVERLAP = 150                                # overlap between consecutive chunks to preserve context
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"          # sentence-transformers model used for embeddings
TOP_K = 4                                          # number of top chunks to retrieve per query
MAX_DISTANCE = 0.6                                 # calibrated: in-context ~0.28–0.38, out-of-context ~0.93–1.03 (see Section 7 for calibration output; adjust if you add new videos)
LLM_PROVIDER = "ollama"                            # "ollama" or "groq", groq for fallback
OLLAMA_MODEL = "llama3.2"                          # local Ollama model tag
GROQ_MODEL = "llama-3.3-70b-versatile"            # Groq hosted model name
GROQ_API_KEY = os.getenv("GROQ_API_KEY")          # set the GROQ_API_KEY environment variable before using Groq
DONT_KNOW = "I don't know"                        # fallback reply when no relevant chunk is found

## Section 2 – Download Transcripts

Provides `get_video_id` to extract an 11-character video ID from any YouTube URL format or a bare ID.
`download_transcript(video_id)` fetches captions and saves them to `TRANSCRIPT_DIR`; already-downloaded files are skipped, and videos without captions are reported and skipped without crashing.

In [3]:
import re
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

# Create transcript directory and a single API object reused for all fetches
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
ytt_api = YouTubeTranscriptApi()


def get_video_id(url_or_id):
    """Return the 11-character video ID from a YouTube URL or a bare ID."""
    url_or_id = url_or_id.strip()
    parsed = urlparse(url_or_id)

    # youtube.com variants: watch?v=, /shorts/, /embed/
    if parsed.netloc in ("www.youtube.com", "youtube.com", "m.youtube.com"):
        qs = parse_qs(parsed.query)
        if "v" in qs:
            vid = qs["v"][0]
            if re.fullmatch(r"[\w-]{11}", vid):
                return vid
        m = re.search(r"/(?:shorts|embed)/(?P<id>[\w-]{11})", parsed.path)
        if m:
            return m.group("id")

    # youtu.be/ID
    if parsed.netloc == "youtu.be":
        vid = parsed.path.lstrip("/").split("?")[0]
        if re.fullmatch(r"[\w-]{11}", vid):
            return vid

    # bare 11-character ID
    if re.fullmatch(r"[\w-]{11}", url_or_id):
        return url_or_id

    raise ValueError("That does not look like a valid YouTube link.")


def download_transcript(video_id):
    """Download and save transcript for video_id. Returns None on skip or error."""
    filepath = TRANSCRIPT_DIR / f"{video_id}.txt"

    # Skip if already downloaded
    if filepath.exists():
        print(f"{video_id}: skipped (already downloaded)")
        return None

    # Try preferred languages first, then accept any available transcript
    snippet_list = None
    try:
        snippet_list = ytt_api.fetch(video_id, languages=TRANSCRIPT_LANGUAGES)
    except Exception:
        pass  # preferred languages not found – try any available transcript

    if snippet_list is None:
        try:
            # Fallback: list all available transcripts and pick the first English one
            transcript_list = ytt_api.list(video_id)
            for t in transcript_list:
                if t.language_code.startswith("en"):
                    snippet_list = ytt_api.fetch(video_id, languages=[t.language_code])
                    break
        except Exception as e:
            print(f"{video_id}: SKIPPED, could not get a transcript ({type(e).__name__})")
            return None

    if snippet_list is None:
        print(f"{video_id}: SKIPPED, no English transcript available")
        return None

    # Write one line per snippet: [start] text
    with open(filepath, "w", encoding="utf-8") as f:
        for snippet in snippet_list:
            text = snippet.text.replace("\n", " ")
            f.write(f"[{snippet.start:.1f}] {text}\n")

    print(f"{video_id}: saved to {filepath}")

## Section 3 – Split Transcripts into Chunks

Reads each saved transcript file, rebuilds the full text, and splits it into overlapping chunks using `RecursiveCharacterTextSplitter`.
Each chunk carries metadata — `video_id`, `chunk_number`, and `start_seconds` — so we can later cite the timestamp in the video.

In [4]:
import re
import bisect
from langchain_text_splitters import RecursiveCharacterTextSplitter

# One splitter instance reused for all videos
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)

_LINE_RE = re.compile(r"^\[(\d+(?:\.\d+)?)\]\s(.+)$")

def read_transcript(path):
    """Return (full_text, line_offsets, line_seconds) for a transcript file."""
    full_text = ""
    line_offsets = []   # character position where each line starts in full_text
    line_seconds = []   # start second of each line

    with open(path, encoding="utf-8") as f:
        for raw in f:
            m = _LINE_RE.match(raw.rstrip("\n"))
            if not m:
                continue  # skip malformed lines
            seconds, text = float(m.group(1)), m.group(2)
            line_offsets.append(len(full_text))
            line_seconds.append(seconds)
            full_text += text + " "

    return full_text, line_offsets, line_seconds


def make_chunks(video_id):
    """Split one video's transcript into metadata-rich chunk dicts."""
    path = TRANSCRIPT_DIR / f"{video_id}.txt"
    full_text, line_offsets, line_seconds = read_transcript(path)

    docs = splitter.create_documents([full_text])
    chunks = []
    for number, doc in enumerate(docs, start=1):
        start_index = max(doc.metadata.get("start_index", 0), 0)
        line_idx = bisect.bisect_right(line_offsets, start_index) - 1
        line_idx = max(line_idx, 0)
        start_seconds = int(line_seconds[line_idx])

        chunks.append({
            "id":            f"{video_id}_{number}",
            "text":          doc.page_content,
            "video_id":      video_id,
            "chunk_number":  number,
            "start_seconds": start_seconds,
        })

    return chunks

## Section 4 – Embeddings and ChromaDB

Encodes every chunk into a vector using `SentenceTransformer` and stores the chunks in a local persistent ChromaDB collection with cosine similarity.
The collection is always rebuilt fresh from the current `all_chunks` so re-running this cell reflects any changes made upstream.

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load the embedding model once
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def embed(texts):
    """Return normalised embeddings as a plain Python list-of-lists."""
    return embed_model.encode(texts, normalize_embeddings=True).tolist()


# Connect to (or create) the local ChromaDB store
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Get existing collection or create it fresh — safe to re-run
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection '{COLLECTION_NAME}' ready ({collection.count()} chunks already stored).")


def ingest_video(video_id):
    """Ensure video_id is ingested into Chroma. Returns True on success, False if no transcript."""
    # Already in Chroma? Nothing to do.
    existing = collection.get(where={"video_id": video_id}, limit=1)
    if existing["ids"]:
        print(f"{video_id}: already in Chroma")
        return True

    # Download transcript (skips silently if file already exists)
    download_transcript(video_id)

    path = TRANSCRIPT_DIR / f"{video_id}.txt"
    if not path.exists():
        return False  # no captions available

    chunks = make_chunks(video_id)
    if not chunks:
        return False

    ids        = [c["id"]   for c in chunks]
    documents  = [c["text"] for c in chunks]
    metadatas  = [
        {
            "video_id":      c["video_id"],
            "chunk_number":  c["chunk_number"],
            "start_seconds": c["start_seconds"],
        }
        for c in chunks
    ]
    embeddings = embed(documents)

    collection.add(ids=ids, documents=documents, embeddings=embeddings, metadatas=metadatas)
    print(f"{video_id}: ingested {len(chunks)} chunks (collection total: {collection.count()})")
    return True

c:\Users\sarth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6316.72it/s]


Collection 'yt_transcripts' ready (50 chunks already stored).


## Section 5 – LLM Interface

Provides a single `ask_llm(messages)` function that routes to either a local Ollama model or the Groq cloud API, depending on `LLM_PROVIDER`.
The temperature is fixed at 0 so answers are deterministic and factual.

**Setup:**
- **Ollama** — download and install from [ollama.com](https://ollama.com), then run `ollama pull llama3.2` once in your terminal to download the model.
- **Groq** — create a free API key at [console.groq.com](https://console.groq.com), then set it as the environment variable `GROQ_API_KEY` before launching Jupyter.

In [6]:
import ollama
from groq import Groq

def ask_llm(messages):
    """Send a message list to the configured LLM and return the reply text."""
    if LLM_PROVIDER == "ollama":
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=messages,
            options={"temperature": 0},
        )
        return response["message"]["content"]

    elif LLM_PROVIDER == "groq":
        if not GROQ_API_KEY:
            raise ValueError(
                "GROQ_API_KEY is not set. Export it as an environment variable "
                "before launching Jupyter, or switch LLM_PROVIDER to 'ollama'."
            )
        client = Groq(api_key=GROQ_API_KEY)
        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            temperature=0,
        )
        return response.choices[0].message.content

    else:
        raise ValueError('LLM_PROVIDER must be "ollama" or "groq".')

In [7]:
# Quick sanity check — should print "LLM is working"
print(ask_llm([{"role": "user", "content": "Reply with exactly: LLM is working"}]))

LLM is working


## Section 6 – Retrieval and Answering

Retrieves the most relevant chunks for a question filtered to a specific video, then asks the LLM to answer using only that context.
`resolve_video` picks the video ID from user input or falls back to `DEFAULT_VIDEO_URL`.
If no chunk is close enough, or the LLM signals it doesn't know, the notebook returns `DONT_KNOW` without hallucinating.

In [8]:
SYSTEM_PROMPT = """You are an expert AI assistant designed to answer questions strictly based on the provided YouTube video transcript context.
Guidelines:
1. Grounding: Answer the user's question using ONLY the information present in the provided transcript snippets. Do not use outside knowledge or invent details.
2. Overview & Summaries: If the user asks for a summary, main takeaways, key pain points, or what the video is about, synthesize the main themes from the transcript snippets into a clear, high-level response.
3. Structure & Formatting: Structure your answers using bullet points, bold key terms, and concise paragraphs for optimal readability.
4. Fallback: If the provided transcript snippets contain no relevant information to answer the question, reply with exactly: I don't know.
5. Tone: Be direct, clear, and factual. Avoid repeating repetitive phrases like "based on the context provided".
"""
_META_KEYWORDS = {                                                        
    "about", "summary", "summarize", "summarise", "overview",             
    "main point", "key point", "pain point", "topic", "cover",            
    "discuss", "explain", "describe", "what does this video",             
}                                                                         

def is_meta_question(question):                                           
    """True if the question asks for a summary or overview."""            
    q = question.lower()                                                  
    return any(kw in q for kw in _META_KEYWORDS)                         


def format_time(seconds):
    """Convert seconds to M:SS or H:MM:SS string."""
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


def youtube_link(video_id, seconds):
    """Return a timestamped YouTube URL."""
    return f"https://www.youtube.com/watch?v={video_id}&t={int(seconds)}s"


def retrieve(question, video_id):
    """Embed the question and return the top-K closest chunks for a specific video."""
    query_embedding = embed([question])
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=TOP_K,
        where={"video_id": video_id},
    )
    chunks = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        chunks.append({
            "text":          results["documents"][0][i],
            "video_id":      meta["video_id"],
            "chunk_number":  meta["chunk_number"],
            "start_seconds": meta["start_seconds"],
            "distance":      results["distances"][0][i],
        })
    return chunks


def says_dont_know(reply):
    """True if the reply is a variant of 'I don't know'."""
    normalised = reply.strip().lower().replace("\u2019", "'")
    return normalised.startswith("i don't know")


def answer_question(question, video_id=None):
    video_id = video_id or DEFAULT_VIDEO_URL
    """Retrieve relevant chunks and ask the LLM; return answer + sources."""
    all_retrieved = retrieve(question, video_id)

    # Meta-questions bypass the distance filter; specific questions are filtered  # ← NEW
    if is_meta_question(question):                                                 # ← NEW
        kept = all_retrieved                                                       # ← NEW
    else:                                                                          # ← NEW
        kept = [c for c in all_retrieved if c["distance"] <= MAX_DISTANCE]

    if not kept:
        return {"answer": DONT_KNOW, "sources": []}

    # Build numbered context blocks
    context_blocks = []
    for i, chunk in enumerate(kept, start=1):
        context_blocks.append(f"[Source {i}]\n{chunk['text']}")
    context = "\n\n".join(context_blocks)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    reply = ask_llm(messages)

    if says_dont_know(reply):
        return {"answer": DONT_KNOW, "sources": []}

    return {"answer": reply, "sources": kept}


def format_sources(sources):
    """Return a markdown bullet list of timestamped YouTube links."""
    lines = []
    for src in sources:
        label = (
            f"Video {src['video_id']} | "
            f"Chunk {src['chunk_number']} | "
            f"{format_time(src['start_seconds'])}"
        )
        link = youtube_link(src["video_id"], src["start_seconds"])
        lines.append(f"- [{label}]({link})")
    return "\n".join(lines)


def resolve_video(url_text):
    """Return a video ID from url_text, or fall back to DEFAULT_VIDEO_URL."""
    url_text = url_text.strip() if url_text else ""
    chosen = url_text if url_text else DEFAULT_VIDEO_URL
    if not chosen:
        raise ValueError("No video link given and DEFAULT_VIDEO_URL is empty.")
    return get_video_id(chosen)


## Section 7 – Test and Tune

Uses the default video to help you calibrate `MAX_DISTANCE`.
Cell 1 shows the best-match distance per question; Cell 2 runs the full answer pipeline.
A good threshold sits above all in-context distances and below all out-of-context distances — update `MAX_DISTANCE` in the config cell and re-run Sections 6 and 7 to apply it.

In [9]:
# Add 2-3 questions that your default video clearly answers
IN_CONTEXT_QUESTIONS = [
    "How did insurance companies historically make money?",
    "What types of insurance policies are mentioned in the video?",
    "Where and how did modern insurance originate?",
    "Where was the insurance conference held on a wet Tuesday night?",
    "What caused property insurance to develop?",
] # e.g. ["What does the speaker say about X?", "How does Y work?"]

OUT_OF_CONTEXT_QUESTIONS = [
    "What is the recipe for baking a chocolate cake?",
    "How do I bake a chocolate cake?",
    "Who won the 2018 FIFA World Cup?",
]

# Resolve and ingest the default video once
video_id = resolve_video("")
ingest_video(video_id)

print(f"Testing against video: {video_id}")
print(f"Current MAX_DISTANCE: {MAX_DISTANCE}\n")

print("In-context questions (distance should be LOW):")
for q in IN_CONTEXT_QUESTIONS:
    best = retrieve(q, video_id)[0]["distance"]
    print(f"  {best:.3f}  |  {q}")

print()
print("Out-of-context questions (distance should be HIGH):")
for q in OUT_OF_CONTEXT_QUESTIONS:
    best = retrieve(q, video_id)[0]["distance"]
    print(f"  {best:.3f}  |  {q}")

qjXgpJpSlCc: already in Chroma
Testing against video: qjXgpJpSlCc
Current MAX_DISTANCE: 0.6

In-context questions (distance should be LOW):
  0.304  |  How did insurance companies historically make money?
  0.383  |  What types of insurance policies are mentioned in the video?
  0.372  |  Where and how did modern insurance originate?
  0.572  |  Where was the insurance conference held on a wet Tuesday night?
  0.381  |  What caused property insurance to develop?

Out-of-context questions (distance should be HIGH):
  0.946  |  What is the recipe for baking a chocolate cake?
  0.935  |  How do I bake a chocolate cake?
  1.034  |  Who won the 2018 FIFA World Cup?


In [10]:
for q in IN_CONTEXT_QUESTIONS + OUT_OF_CONTEXT_QUESTIONS:
    result = answer_question(q, video_id)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    if result["sources"]:
        print(format_sources(result["sources"]))
    print("-" * 60)

Q: How did insurance companies historically make money?
A: **Historical Revenue Streams for Insurance Companies**

According to the provided transcript snippets, insurance companies historically made money through the following methods:

* **Assessing risk and offering premiums**: Insurance companies evaluate the risk and decide whether it is worth the gamble. For example, Jim believes Bob won't lose his phone, and if he finds 100 more people willing to give him 10 bucks each to cover their phones, he has 1,000 dollars.
* **Investing premiums**: Modern day insurance companies invest the premium from thousands of policies in another financial product. This allows them to generate revenue even if they pay out more claims than they make in policy premiums.
* **Spreading risk**: Insurance helps spread risk by taking a risk from an individual and spreading it around a community. This allows individuals to go about their personal or business life without financial ruin.

These historical rev

### How to pick `MAX_DISTANCE`

After running **Cell 1**, look at the two groups of numbers:

- **In-context distances** should be **low** (roughly < 0.5) — these chunks are genuinely about your video topics.
- **Out-of-context distances** should be **high** (roughly > 0.6) — these questions have nothing to do with your videos.

**Choose `MAX_DISTANCE` somewhere between the two groups**, for example:

| In-context best distances | Out-of-context best distances | Suggested MAX_DISTANCE |
|:---:|:---:|:---:|
| 0.25 – 0.40 | 0.65 – 0.80 | 0.55 |
| 0.35 – 0.50 | 0.60 – 0.75 | 0.58 |

Once you have picked a value, **update `MAX_DISTANCE` in the config cell (Section 1)** and re-run Sections 6 and 7.
If the in-context and out-of-context distances overlap, consider trying a stronger embedding model or adding more videos.

## Section 8 – Terminal Chat Loop

An interactive loop that lets you query the chatbot inside the notebook.
At startup you can paste any YouTube link or press Enter to use `DEFAULT_VIDEO_URL`; the first use of a new video takes a few seconds while it is downloaded and ingested.
Each question is answered independently — the bot has no memory of earlier questions.
Run Section 9's UI cell only after typing `exit` to stop this loop.

In [11]:
print('Chat started. Type "exit" to stop.\n')

# --- Video selection with retry ---
video_id = None
while video_id is None:
    url_input = input("Paste a YouTube link or ID (or press Enter for the default): ").strip()
    try:
        video_id = resolve_video(url_input)
    except ValueError as e:
        print(f"  ⚠  {e}  Please try again.\n")

print(f"Loading video {video_id}...")
success = ingest_video(video_id)
if not success:
    print("This video has no captions I can use — the loop will exit.")
    video_id = None  # guard below will print nothing

# --- Q&A loop ---
if video_id:
    print(f"Ready. Asking about video: {video_id}\n")
    while True:
        user_input = input("You: ").strip()

        if user_input.lower() == "exit":
            print("Bye!")
            break

        if not user_input:
            continue

        result = answer_question(user_input, video_id)
        print(f"Bot: {result['answer']}")

        if result["sources"]:
            print("Sources:")
            print(format_sources(result["sources"]))

        print()

Chat started. Type "exit" to stop.

Loading video qjXgpJpSlCc...
qjXgpJpSlCc: already in Chroma
Ready. Asking about video: qjXgpJpSlCc

Bye!


## Section 9 – Gradio Chat UI
Launches a custom dark-themed Q&A interface. Paste a YouTube link and press **Enter** or **Load** — a status pill confirms when the video is ready. Ask questions in the chat below; answers stream word-by-word and source timestamps appear as clickable chips.
If the port is busy, run `demo.close()` in a new cell, then re-run this one.


In [12]:
import gradio as gr

# Custom dark theme definition
custom_theme = gr.themes.Base(
    primary_hue="indigo",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "monospace"],
).set(
    body_background_fill="#0b0b0f",
    body_text_color="#f1f1f3",
    block_background_fill="#14141b",
    block_border_color="#232330",
    block_title_text_color="#8b8ba7",
    input_background_fill="#14141b",
    button_primary_background_fill="#6366f1",
    button_primary_text_color="#ffffff",
    code_background_fill="#1a1a24",
)

current_active_video = DEFAULT_VIDEO_URL

def load_video_and_ingest(url_input):
    global current_active_video
    target = url_input.strip() if url_input and url_input.strip() else DEFAULT_VIDEO_URL
    try:
        vid_id = resolve_video(target)
        success = ingest_video(vid_id)
        if success:
            current_active_video = vid_id
            return (
                f"✅ **Video ready:** `{vid_id}` — You can now ask questions below.",
                f"https://img.youtube.com/vi/{vid_id}/hqdefault.jpg"
            )
        else:
            return (
                f"⚠️ **Could not load transcript for:** `{vid_id}`. Please check if captions exist.",
                ""
            )
    except Exception as e:
        return f"❌ **Error:** {e}", ""

def chat_fn(message, history):
    global current_active_video
    if message.strip().lower() == "exit":
        return "Session ended. Restart the cell or refresh the UI to chat again."
    if not current_active_video:
        current_active_video = resolve_video(DEFAULT_VIDEO_URL)
    try:
        result = answer_question(message, current_active_video)
    except Exception as e:
        return f"Something went wrong: {e}"

    answer = result["answer"]
    sources = result["sources"]

    if not sources:
        return answer

    return answer + "\n\n**Sources:**\n" + format_sources(sources)

custom_css = """
.main-title { font-size: 24px; font-weight: 700; color: #ffffff; margin-bottom: 4px; }
.sub-title { font-size: 14px; color: #94a3b8; margin-bottom: 16px; }
.status-pill { padding: 8px 12px; background: #1a1a24; border: 1px solid #232330; border-radius: 8px; font-size: 13px; }
"""

with gr.Blocks(title="YouTube Video Q&A") as demo:
    gr.HTML("""
    <div style='margin-bottom: 16px;'>
        <div class='main-title'>🎬 YouTube Video Q&A</div>
        <div class='sub-title'>Ask any question about a YouTube video. Answers are strictly sourced from its transcript.</div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            url_box = gr.Textbox(
                value=DEFAULT_VIDEO_URL,
                label="YouTube Video Link or ID",
                placeholder="Paste video URL or ID and hit Enter or click Load...",
                lines=1,
            )
            load_btn = gr.Button("Load Video", variant="primary")
        with gr.Column(scale=2):
            status_box = gr.Markdown(
                value=f"✅ **Default video active:** `{DEFAULT_VIDEO_URL}`",
                elem_classes=["status-pill"]
            )
            thumb_img = gr.Image(
                value=f"https://img.youtube.com/vi/{DEFAULT_VIDEO_URL}/hqdefault.jpg" if DEFAULT_VIDEO_URL else None,
                show_label=False,
                interactive=False,
                height=120
            )

    load_btn.click(
        fn=load_video_and_ingest,
        inputs=[url_box],
        outputs=[status_box, thumb_img]
    )
    url_box.submit(
        fn=load_video_and_ingest,
        inputs=[url_box],
        outputs=[status_box, thumb_img]
    )

    chatbot = gr.ChatInterface(
        fn=chat_fn,
        textbox=gr.Textbox(placeholder="Ask a question about this video... (Press Enter to send)", lines=1),
    )

demo.launch(inbrowser=True, theme=custom_theme, css=custom_css)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


qjXgpJpSlCc: already in Chroma
xH_IlU7nIRs: saved to transcripts\xH_IlU7nIRs.txt
xH_IlU7nIRs: ingested 8 chunks (collection total: 58)
